# 01 — Data Audit

**Immunization Defaulter Risk Engine** · Dr. Erick Kiprotich Yegon

Profiles the 12-table PostgreSQL source schema and the merged analytical dataset.

**Objectives**
- Verify PostgreSQL source table row counts and schema
- Audit the merged `analytical_dataset.parquet` (6,864 patients × 154 columns)
- Inspect immunization backbone completeness (`iz` table)
- Confirm CHW area linkage quality and cardinality
- Document known data issues that inform cleaning decisions

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR    = PROJECT_ROOT / "reports"

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.3f}".format)
print("Project root:", PROJECT_ROOT)

## 1. PostgreSQL source table inventory

In [ ]:
from src.ingestion.db import get_engine
from sqlalchemy import text

engine = get_engine()

SOURCE_TABLES = [
    "iz", "active_chps", "supervision", "homevisit", "population",
    "pnc", "preg_reg", "preg_reg2", "preg_visit", "preg_visit2", "fp", "refill",
]

records = []
with engine.connect() as conn:
    for tbl in SOURCE_TABLES:
        try:
            n_rows = conn.execute(text(f"SELECT COUNT(*) FROM {tbl}")).scalar()
            n_cols = conn.execute(
                text("SELECT COUNT(*) FROM information_schema.columns "
                     "WHERE table_name = :t AND table_schema = 'public'"),
                {"t": tbl}
            ).scalar()
            records.append({"table": tbl, "raw_rows": n_rows, "columns": n_cols})
        except Exception as e:
            records.append({"table": tbl, "raw_rows": f"ERROR: {e}", "columns": None})

src_df = pd.DataFrame(records)
src_df["raw_rows_fmt"] = src_df["raw_rows"].apply(
    lambda x: f"{x:,}" if isinstance(x, int) else x
)
src_df[["table", "raw_rows_fmt", "columns"]].rename(columns={"raw_rows_fmt": "raw_rows"})

## 2. Analytical dataset — post-ETL merged parquet

In [ ]:
df = pd.read_parquet(DATA_PROCESSED / "analytical_dataset.parquet")

print(f"Shape            : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Unique patients  : {df['patient_id'].nunique():,}")
print(f"Unique CHW areas : {df['contact_parent_id'].nunique():,}")
print(f"Unique counties  : {df['county'].nunique()}")
print(f"Months covered   : {df['month_clean'].nunique()} ({df['month_clean'].min()} – {df['month_clean'].max()})")
print(f"\nTarget (is_defaulter):")
print(df["is_defaulter"].value_counts(normalize=True).rename(index={0: "Non-defaulter", 1: "Defaulter"}).round(4))

df[["patient_id", "patient_age_in_months", "patient_sex", "county",
    "month_clean", "is_defaulter"]].head(5)

## 3. Completeness audit — analytical dataset

In [ ]:
audit = pd.DataFrame({
    "column"      : df.columns,
    "missing_pct" : (df.isna().mean() * 100).round(1).values,
    "n_unique"    : df.nunique(dropna=True).values,
    "dtype"       : df.dtypes.astype(str).values,
}).sort_values("missing_pct", ascending=False).reset_index(drop=True)

print(f"Columns with >80% missing : {(audit.missing_pct > 80).sum()}")
print(f"Columns with 0%  missing  : {(audit.missing_pct == 0).sum()}")
print(f"Feature matrix columns    : {df.shape[1]}")
audit.head(25)

In [ ]:
# Missingness heatmap
fig, ax = plt.subplots(figsize=(9, 4))
audit["missing_pct"].plot(kind="hist", bins=20, ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Missingness distribution across 154 analytical columns", fontsize=13, fontweight="bold")
ax.set_xlabel("Column missingness (%)")
ax.set_ylabel("Column count")
ax.axvline(80, color="red", linestyle="--", alpha=0.6, label="80% threshold")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Immunization backbone — `iz` table audit

In [ ]:
# Vaccine column presence rates
vax_cols = [c for c in df.columns if c.startswith("has_") and c not in
            ["has_good_immunization_status", "has_been_referred",
             "has_signs_of_delayed_milestones", "has_received_vaccines_due"]]

vax_audit = pd.DataFrame({
    "vaccine"    : vax_cols,
    "received_n" : [int(pd.to_numeric(df[c], errors="coerce").sum()) for c in vax_cols],
    "pct_received": [
        round(pd.to_numeric(df[c], errors="coerce").mean() * 100, 1)
        for c in vax_cols
    ],
    "pct_null"   : [(df[c].isna().mean() * 100).round(1) for c in vax_cols],
}).sort_values("pct_received", ascending=False)

print(f"Vaccine history columns: {len(vax_cols)}")
vax_audit.head(20)

In [ ]:
# Age distribution of patients
ages = pd.to_numeric(df["patient_age_in_months"], errors="coerce").dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(ages, bins=24, color="steelblue", edgecolor="white")
axes[0].set_title("Patient age distribution (months)", fontweight="bold")
axes[0].set_xlabel("Age (months)")
axes[0].set_ylabel("Patients")

sex_counts = df["patient_sex"].str.lower().value_counts(dropna=False)
axes[1].bar(sex_counts.index.astype(str), sex_counts.values, color=["steelblue","coral","grey"][:len(sex_counts)])
axes[1].set_title("Patient sex distribution", fontweight="bold")
axes[1].set_ylabel("Patients")

plt.tight_layout()
plt.show()

print("Age stats (months):")
print(ages.describe().round(1).to_string())

## 5. Geographic and temporal coverage

In [ ]:
# County-level patient counts
county_counts = df.groupby("county").agg(
    patients     = ("patient_id", "nunique"),
    chw_areas    = ("contact_parent_id", "nunique"),
    defaulter_rate = ("is_defaulter", "mean"),
).sort_values("patients", ascending=False).reset_index()

county_counts["defaulter_rate"] = (county_counts["defaulter_rate"] * 100).round(1)
print(f"Counties represented: {len(county_counts)}")
county_counts.head(20)

In [ ]:
# Temporal coverage — patients by month
monthly = df.groupby("month_clean").agg(
    patients   = ("patient_id", "count"),
    defaulters = ("is_defaulter", "sum"),
).reset_index().sort_values("month_clean")

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(range(len(monthly)), monthly["patients"], label="Total", color="steelblue", alpha=0.7)
ax.bar(range(len(monthly)), monthly["defaulters"], label="Defaulters", color="coral")
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly["month_clean"], rotation=45, ha="right", fontsize=8)
ax.set_title("Patient records by reporting month", fontsize=13, fontweight="bold")
ax.set_ylabel("Records")
ax.legend()
plt.tight_layout()
plt.show()

## 6. CHW context linkage quality

In [ ]:
chw_context_cols = [
    "chw_supervision_frequency", "chw_immunization_competency_pct",
    "chw_overall_assessment_pct", "chw_has_all_tools", "chw_has_ppe",
    "chw_workload_u2", "monthly_homevisit_rate", "months_since_last_supervision",
]

linkage = pd.DataFrame({
    "feature"     : chw_context_cols,
    "null_pct"    : [(df[c].isna().mean() * 100).round(1) if c in df.columns else 100.0
                    for c in chw_context_cols],
    "linked_n"    : [(df[c].notna().sum()) if c in df.columns else 0
                    for c in chw_context_cols],
    "mean_value"  : [round(pd.to_numeric(df[c], errors="coerce").mean(), 3)
                    if c in df.columns else None for c in chw_context_cols],
})

print("CHW context linkage (from supervision + active_chps + homevisit + population joins):")
linkage

## 7. Audit conclusions

| Finding | Detail |
|---|---|
| **Source tables** | 12 PostgreSQL tables; 3 operational tables exceed 1M rows (aggregated server-side via SQL) |
| **Analytical dataset** | 6,864 patient records post-deduplication; 16.5% defaulter prevalence |
| **Vaccine backbone** | BCG near-universal (>95%); pentavalent series drops off for Penta-3 (~72%); malaria vaccines zero outside endemic zones |
| **CHW linkage** | Supervision and workload features linked for majority of patients; maternal join at 0% (step 8 — known gap in maternal registration data) |
| **Temporal coverage** | Multiple months across 2024–2025; no single dominant month (low temporal bias risk) |
| **Geographic** | 47 counties represented; defaulter rates vary substantially by county — geographic encoding important |